# Sequence Mining: Complete Notebook
> **Course:** Machine Learning / Business Intelligence
> **Goal:** Discover temporally ordered patterns in web sessions and purchase sequences

---

## What will you learn?
1. The fundamental difference between **Association Rules** and **Sequence Mining**
2. Preprocess a **raw event log** into a sequence database
3. Mine frequent sequences with **PrefixSpan**
4. Compute **Sequential Support** and **Sequential Confidence**
5. Build and interpret a **Transition Matrix**
6. Apply the results to **web funnel analysis** and **cross-sell recommendations**

---
### Why does order matter?

| Technique | Input | Does order matter? | Question it answers |
|---|---|---|---|
| Association Rules | `{leche, pan, mantequilla}` | No | What is bought **together**? |
| Sequence Mining | `[home → products → cart → checkout]` | Yes | What is done **after** what? |

> **Example:** `[laptop → mouse]` and `[mouse → laptop]` are **two different patterns** in SM.
> In AR, `{laptop, mouse}` is **the same itemset** regardless of order.

## 1. Installation and Setup

In [ ]:
!pip install prefixspan --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from prefixspan import PrefixSpan
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#18181b',
    'axes.facecolor':   '#27272a',
    'axes.edgecolor':   '#3f3f46',
    'text.color':       '#e4e4e7',
    'axes.labelcolor':  '#a1a1aa',
    'xtick.color':      '#71717a',
    'ytick.color':      '#71717a',
    'grid.color':       '#3f3f46',
    'grid.alpha':        0.4,
    'axes.titlesize':    13,
    'axes.titlepad':     10,
})

BLUE    = '#3b82f6'
EMERALD = '#10b981'
AMBER   = '#f59e0b'
PURPLE  = '#8b5cf6'
PINK    = '#ec4899'
RED     = '#ef4444'
CYAN    = '#06b6d4'
ZINC    = '#71717a'

np.random.seed(42)
print('Libraries loaded successfully')

---
## 2. Datasets

We will work with **two different business contexts**:

| # | Dataset | Sequence unit | Event | Business question |
|---|---|---|---|---|
| A | **Web Navigation** | User session | Page visited | What are the paths to checkout? |
| B | **Purchase History** | Customer | Category purchased | What does the customer buy after a laptop? |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DATASET A: Web Navigation Sessions (200 sessions)
# ══════════════════════════════════════════════════════════════════

# Navigation patterns with their probabilities
NAVIGATION_PATTERNS = [
    (['home', 'products', 'cart', 'checkout'],          0.18),
    (['home', 'products', 'checkout'],                  0.12),
    (['home', 'search', 'products', 'cart', 'checkout'],0.10),
    (['home', 'products', 'cart'],                      0.09),
    (['home', 'blog', 'products', 'cart', 'checkout'],  0.07),
    (['home', 'products', 'cart', 'payment', 'confirm'],0.07),
    (['home', 'about', 'contact'],                      0.05),
    (['home', 'search', 'products', 'checkout'],        0.06),
    (['home', 'login', 'products', 'cart', 'checkout'], 0.06),
    (['home', 'pricing', 'products', 'checkout'],       0.05),
    (['home', 'blog', 'about'],                         0.04),
    (['home', 'products', 'pricing', 'checkout'],       0.04),
    (['home', 'blog', 'products'],                      0.04),
    (['home', 'about', 'pricing', 'products'],          0.03),
]

def sample_sessions(patterns, n=200):
    probs = np.array([p for _, p in patterns])
    probs = probs / probs.sum()
    chosen = np.random.choice(len(patterns), size=n, p=probs)
    sessions = []
    for idx in chosen:
        base = patterns[idx][0].copy()
        # Add occasional random variation
        if np.random.random() < 0.15:
            extra = np.random.choice(['search', 'blog', 'pricing', 'about'])
            insert_at = np.random.randint(1, max(1, len(base) - 1))
            base.insert(insert_at, extra)
        sessions.append(base)
    return sessions

web_sessions = sample_sessions(NAVIGATION_PATTERNS, 200)

# ══════════════════════════════════════════════════════════════════
# DATASET B: Customer Purchase History (150 customers)
# ══════════════════════════════════════════════════════════════════

PURCHASE_PATTERNS = [
    (['laptop', 'mouse', 'keyboard', 'laptop_bag'],   0.14),
    (['phone', 'case', 'charger', 'earbuds'],          0.13),
    (['laptop', 'mouse', 'headphones'],                0.10),
    (['tablet', 'keyboard', 'stylus'],                 0.09),
    (['phone', 'charger', 'case', 'screen_protector'], 0.09),
    (['laptop', 'laptop_bag', 'mouse', 'keyboard'],    0.08),
    (['phone', 'earbuds', 'charger'],                  0.08),
    (['tablet', 'case', 'keyboard'],                   0.07),
    (['laptop', 'headphones', 'mouse'],                0.07),
    (['phone', 'case', 'charger', 'cable'],            0.06),
    (['laptop', 'mouse', 'keyboard', 'webcam'],        0.05),
    (['phone', 'charger', 'screen_protector'],         0.04),
]

purchase_sequences = sample_sessions(PURCHASE_PATTERNS, 150)

print(f'Dataset A: web sessions      : {len(web_sessions):4d} sessions')
print(f'Dataset B: customer purchases: {len(purchase_sequences):4d} sequences')
print()
print('Example web sessions:')
for s in web_sessions[:3]:
    print(f'  {" → ".join(s)}')
print('Example purchase sequences:')
for s in purchase_sequences[:3]:
    print(f'  {" → ".join(s)}')

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
def eda_summary(sequences, name):
    lengths = [len(s) for s in sequences]
    all_events = [e for s in sequences for e in s]
    unique_events = sorted(set(all_events))
    event_counts = pd.Series(all_events).value_counts()
    event_support = event_counts / len(sequences)
    print(f'{'═'*55}')
    print(f'  {name}')
    print(f'{'═'*55}')
    print(f'  Total sequences      : {len(sequences)}')
    print(f'  Unique events        : {len(unique_events)}')
    print(f'  Average length       : {np.mean(lengths):.1f} events/sequence')
    print(f'  Min / max length     : {min(lengths)} / {max(lengths)}')
    print(f'  Density              : {sum(lengths)/(len(sequences)*len(unique_events)):.3f}')
    print(f'  Top 5 events         : {list(event_support.head(5).index)}')
    return event_support

web_support  = eda_summary(web_sessions,       'DATASET A: Web Navigation')
print()
purch_support = eda_summary(purchase_sequences, 'DATASET B: Customer Purchases')

In [ ]:
# Colors per domain
WEB_COLORS = {
    'home': EMERALD, 'products': BLUE, 'cart': AMBER, 'checkout': PURPLE,
    'search': CYAN, 'payment': '#a855f7', 'confirm': '#22c55e',
    'login': PINK, 'pricing': '#f97316', 'blog': ZINC, 'about': ZINC, 'contact': ZINC,
}
PURCH_COLORS = {
    'laptop': BLUE, 'mouse': EMERALD, 'keyboard': EMERALD, 'laptop_bag': AMBER,
    'headphones': PURPLE, 'webcam': CYAN, 'phone': '#f97316', 'case': PINK,
    'charger': '#a855f7', 'earbuds': '#22c55e', 'screen_protector': ZINC,
    'cable': ZINC, 'tablet': '#0ea5e9', 'stylus': AMBER,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Support per Event: Individual Frequency in Each Dataset')

for ax, support, colors, title in [
    (axes[0], web_support,   WEB_COLORS,   'Web Pages (Dataset A)'),
    (axes[1], purch_support, PURCH_COLORS, 'Purchase Categories (Dataset B)'),
]:
    top = support.head(12)
    bar_colors = [colors.get(e, ZINC) for e in top.index]
    ax.barh(top.index[::-1], top.values[::-1], color=bar_colors[::-1], edgecolor='none', height=0.65)
    ax.set_xlabel('Support (fraction of sequences)')
    ax.set_title(title)
    ax.grid(axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Sequence length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Sequence Length Distribution')

for ax, seqs, color, label in [
    (axes[0], web_sessions,       BLUE,    'Web Sessions'),
    (axes[1], purchase_sequences, '#f97316','Purchase Sequences'),
]:
    lengths = [len(s) for s in seqs]
    ax.hist(lengths, bins=range(1, max(lengths)+2), color=color, alpha=0.85,
            edgecolor='#18181b', rwidth=0.8)
    ax.axvline(np.mean(lengths), color=AMBER, linewidth=2, linestyle='--',
               label=f'mean = {np.mean(lengths):.1f}')
    ax.set_xlabel('Sequence length (# events)')
    ax.set_ylabel('Frequency')
    ax.set_title(label)
    ax.legend()
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

---
## 4. Preprocessing: From the Raw Log to the Sequence Database

In practice, data **never arrives already processed**. A real system hands you a log like this one:

```
session_id | timestamp           | page
S003       | 2024-03-01 14:02:01 | home
S001       | 2024-03-01 14:02:14 | home       <- interleaved!
S003       | 2024-03-01 14:02:33 | products
S001       | 2024-03-01 14:02:45 | products
...
```

### Preprocessing pipeline for SM
1. **Raw log**: interleaved records with timestamps
2. **Grouping by session**: sort events by timestamp for each user
3. **Filtering**: remove sessions of length 1
4. **Sequence database**: a list of lists, ready for the algorithm

> **Key difference vs AR:** no binary encoding is needed. Order is preserved as is.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: Simulate a raw event log (interleaved, with timestamps)
# ══════════════════════════════════════════════════════════════════
import datetime

def build_raw_log(sequences, session_prefix='S', start_time=None, interval_secs=(15, 180)):
    """Build a raw event log from assembled sequences."""
    if start_time is None:
        start_time = datetime.datetime(2024, 3, 1, 9, 0, 0)
    records = []
    current_time = start_time
    for i, seq in enumerate(sequences[:10]):          # first 10 sessions
        session_id = f'{session_prefix}{i+1:03d}'
        for event in seq:
            records.append({'session_id': session_id, 'timestamp': current_time, 'event': event})
            delta = np.random.randint(*interval_secs)
            current_time += datetime.timedelta(seconds=delta)
    # Shuffle records (simulate a real interleaved log)
    df = pd.DataFrame(records).sample(frac=1, random_state=7).reset_index(drop=True)
    return df

raw_log_web = build_raw_log(web_sessions, 'S')
print('STEP 1: Raw log (20 records, interleaved):')
print(raw_log_web.head(20).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: Group by session and sort by time
# ══════════════════════════════════════════════════════════════════
def assemble_sessions(raw_log):
    """Group events by session and sort by timestamp."""
    assembled = (
        raw_log
        .sort_values(['session_id', 'timestamp'])
        .groupby('session_id')['event']
        .apply(list)
        .reset_index()
    )
    assembled.columns = ['session_id', 'sequence']
    assembled['length'] = assembled['sequence'].apply(len)
    return assembled

assembled_web = assemble_sessions(raw_log_web)
print('STEP 2: Assembled sessions (grouped and sorted):')
for _, row in assembled_web.iterrows():
    arrow_seq = ' → '.join(row['sequence'])
    print(f"  {row['session_id']}  ({row['length']} events)  {arrow_seq}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: Filter out sequences that are too short
# ══════════════════════════════════════════════════════════════════
def filter_sequences(assembled, min_length=2):
    before = len(assembled)
    filtered = assembled[assembled['length'] >= min_length].copy()
    removed  = before - len(filtered)
    print(f'Sessions before filtering   : {before}')
    print(f'Removed (length < {min_length})         : {removed}')
    print(f'Sessions kept               : {len(filtered)}')
    return filtered

# Artificially add a few length-1 sessions to demonstrate the filter
short_sessions = pd.DataFrame([
    {'session_id': 'S_X1', 'sequence': ['home'], 'length': 1},
    {'session_id': 'S_X2', 'sequence': ['products'], 'length': 1},
])
assembled_web_ext = pd.concat([assembled_web, short_sessions], ignore_index=True)

print('STEP 3: Sequence filtering:')
print()
filtered_web = filter_sequences(assembled_web_ext, min_length=2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: Sequence database (final format)
# ══════════════════════════════════════════════════════════════════
# For SM algorithms, the format is simply a list of lists
# NO binary matrix or numeric encoding is needed

web_db     = web_sessions       # Full Dataset A (already clean)
purchase_db = purchase_sequences # Full Dataset B

print('STEP 4: Final format of the sequence database:')
print('(list of lists: each sublist is an ordered session)')
print()
print('web_db[:5] =')
for seq in web_db[:5]:
    print(f'  {seq}')
print()
print('purchase_db[:5] =')
for seq in purchase_db[:5]:
    print(f'  {seq}')
print()
print('Ready for PrefixSpan: no binarization, no numeric encoding')

---
## 5. Mining with PrefixSpan

### How does PrefixSpan work?
1. Find all **frequent prefixes** of length 1
2. For each prefix `α`, build the **projected database**: only the **suffixes** of the sequences that contain `α`
3. Recursively mine the projected database to extend the prefix
4. Guarantee: **every pattern discovered is frequent by construction** (no false candidates are generated)

```
Projected database for [home]:
  S001: [products, cart, checkout]  <- suffix after 'home'
  S002: [products, checkout]
  S003: [search, products, cart, checkout]
  ...
```

In [ ]:
# ── Helper function for projection ───────────────────────────────
def get_projected_db(sequences, prefix_event, show=6):
    """Show the projected database for a given prefix."""
    print(f'Projected database for prefix [{prefix_event}]:')
    print(f'{"─"*55}')
    count = 0
    for i, seq in enumerate(sequences):
        if prefix_event in seq:
            idx = seq.index(prefix_event)
            suffix = seq[idx+1:]
            if count < show:
                suffix_str = ' → '.join(suffix) if suffix else '∅ (empty)'
                print(f'  S{i+1:03d}: [{prefix_event}] | {suffix_str}')
            count += 1
    if count > show:
        print(f'  ... ({count - show} more sessions)')
    print(f'Total sequences with [{prefix_event}]: {count} -> support = {count/len(sequences):.2f}')

get_projected_db(web_db, 'home')
print()
get_projected_db(web_db, 'products')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Mine Dataset A: Web Navigation
# ══════════════════════════════════════════════════════════════════
MIN_SUPPORT_WEB = 0.30   # <- try 0.20, 0.40, 0.50
MAX_LEN_WEB     = 4

ps_web = PrefixSpan(web_db)
min_count_web = int(MIN_SUPPORT_WEB * len(web_db))
results_web_raw = ps_web.frequent(min_count_web, closed=False)

# Convert to DataFrame
df_web = pd.DataFrame(
    [(count, pattern) for count, pattern in results_web_raw if len(pattern) <= MAX_LEN_WEB],
    columns=['count', 'pattern']
)
df_web['support']  = df_web['count'] / len(web_db)
df_web['length']   = df_web['pattern'].apply(len)
df_web['pattern_str'] = df_web['pattern'].apply(lambda p: ' → '.join(p))
df_web = df_web.sort_values('support', ascending=False)

print(f'Dataset A: Web Navigation')
print(f'Frequent patterns found (support >= {MIN_SUPPORT_WEB}): {len(df_web)}')
print()
print(df_web[['pattern_str', 'support', 'length']].head(15).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Mine Dataset B: Purchase History
# ══════════════════════════════════════════════════════════════════
MIN_SUPPORT_PURCH = 0.25
MAX_LEN_PURCH     = 4

ps_purch = PrefixSpan(purchase_db)
min_count_purch = int(MIN_SUPPORT_PURCH * len(purchase_db))
results_purch_raw = ps_purch.frequent(min_count_purch, closed=False)

df_purch = pd.DataFrame(
    [(count, pattern) for count, pattern in results_purch_raw if len(pattern) <= MAX_LEN_PURCH],
    columns=['count', 'pattern']
)
df_purch['support']     = df_purch['count'] / len(purchase_db)
df_purch['length']      = df_purch['pattern'].apply(len)
df_purch['pattern_str'] = df_purch['pattern'].apply(lambda p: ' → '.join(p))
df_purch = df_purch.sort_values('support', ascending=False)

print(f'Dataset B: Purchase History')
print(f'Frequent patterns found (support >= {MIN_SUPPORT_PURCH}): {len(df_purch)}')
print()
print(df_purch[['pattern_str', 'support', 'length']].head(15).to_string(index=False))

---
## 6. Sequential Metrics

### Sequential Support
$$\text{support}([A \to B]) = \frac{|\{\text{sequences that contain A before B}\}|}{|\text{total sequences}|}$$

**Gap-independent**: `[A...B]` counts even if there are other events between A and B.

### Sequential Confidence
$$\text{conf}([A] \Rightarrow [B]) = \frac{\text{support}([A \to B])}{\text{support}([A])}$$

Given that A occurred, how likely is it that B occurs afterwards?

In [ ]:
def is_subsequence(pattern, sequence):
    """Check whether 'pattern' is a subsequence of 'sequence' (gaps allowed)."""
    pi = 0
    for event in sequence:
        if pi < len(pattern) and event == pattern[pi]:
            pi += 1
    return pi == len(pattern)

def seq_support(pattern, sequences):
    count = sum(1 for s in sequences if is_subsequence(pattern, s))
    return count / len(sequences)

def seq_confidence(antecedent, consequent_event, sequences):
    """conf([antecedent] => [consequent_event]): looks for the event after the antecedent."""
    pattern_together = antecedent + [consequent_event]
    sup_together = seq_support(pattern_together, sequences)
    sup_ant      = seq_support(antecedent, sequences)
    return sup_together / sup_ant if sup_ant > 0 else 0

# ── Manual check: Dataset A ──────────────────────────────────────
print('=== DATASET A METRICS: Web Navigation ===')
test_cases_web = [
    (['home'], 'products'),
    (['home'], 'checkout'),
    (['home', 'products'], 'cart'),
    (['home', 'products', 'cart'], 'checkout'),
    (['products', 'cart'], 'checkout'),
]
for ant, con in test_cases_web:
    sup  = seq_support(ant + [con], web_db)
    conf = seq_confidence(ant, con, web_db)
    print(f'  [{" → ".join(ant)}] ⟹ [{con}]')
    print(f'    support = {sup:.3f}  |  confidence = {conf:.3f}')
    print()

print()
print('=== DATASET B METRICS: Purchase History ===')
test_cases_purch = [
    (['laptop'], 'mouse'),
    (['phone'],  'charger'),
    (['laptop', 'mouse'], 'keyboard'),
    (['phone', 'charger'], 'earbuds'),
]
for ant, con in test_cases_purch:
    sup  = seq_support(ant + [con], purchase_db)
    conf = seq_confidence(ant, con, purchase_db)
    print(f'  [{" → ".join(ant)}] ⟹ [{con}]')
    print(f'    support = {sup:.3f}  |  confidence = {conf:.3f}')
    print()

---
## 7. Transition Matrix

The **sequential confidence between pairs of events** can be visualized as a transition matrix:

$$M[A][B] = P(B \text{ occurs after } A \text{ in the same sequence})$$

This matrix is the basis of **first-order Markov models** for web navigation.

In [ ]:
def build_transition_matrix(sequences, top_n=10):
    """Build the matrix of transition probabilities among the top_n events."""
    all_events = [e for s in sequences for e in s]
    top_events = pd.Series(all_events).value_counts().head(top_n).index.tolist()
    n = len(sequences)
    matrix = pd.DataFrame(0.0, index=top_events, columns=top_events)
    for from_ev in top_events:
        for to_ev in top_events:
            count = sum(1 for seq in sequences
                        if from_ev in seq and to_ev in seq[seq.index(from_ev)+1:])
            matrix.loc[from_ev, to_ev] = count / n
    return matrix

# ── Dataset A: Web Navigation ─────────────────────────────────────
print('Computing transition matrix for web navigation...')
transition_web   = build_transition_matrix(web_db, top_n=9)
print('Computing transition matrix for purchases...')
transition_purch = build_transition_matrix(purchase_db, top_n=10)
print('Done')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Sequential Confidence Matrices\n(cell [i,j] = P(column j occurs after row i))')

for ax, matrix, title in [
    (axes[0], transition_web,   'Dataset A: Web Navigation'),
    (axes[1], transition_purch, 'Dataset B: Customer Purchases'),
]:
    # Mask the diagonal
    mask = np.eye(len(matrix), dtype=bool)
    sns.heatmap(
        matrix, ax=ax, mask=mask,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.4, linecolor='#18181b',
        annot=True, fmt='.2f', annot_kws={'size': 9}
    )
    ax.set_title(title)
    ax.set_xlabel('Next event (column)')
    ax.set_ylabel('From event (row)')
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.show()

print('Interpretation: high values (dark blue) = high probability that the column')
print('occurs AFTER the row. E.g. P(checkout | home -> products -> cart) is the sequential confidence.')

---
## 8. Visualizations

In [ ]:
# ── Top patterns by support: Dataset A ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top Frequent Patterns by Support')

for ax, df, color, n_top, title in [
    (axes[0], df_web.query('length >= 2'),   BLUE,     12, 'Dataset A: Web Navigation'),
    (axes[1], df_purch.query('length >= 2'), '#f97316', 12, 'Dataset B: Customer Purchases'),
]:
    top = df.nlargest(n_top, 'support')
    ax.barh(top['pattern_str'][::-1], top['support'][::-1],
            color=color, alpha=0.85, edgecolor='none', height=0.65)
    ax.set_xlabel('Support')
    ax.set_title(title)
    ax.grid(axis='x')
    # Annotate pattern length
    for _, row in top.iterrows():
        ax.text(row['support'] + 0.005, list(top['pattern_str'])[::-1].index(row['pattern_str']),
                f"L={row['length']}", va='center', fontsize=7, color=AMBER)

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution of patterns by length ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Number of Patterns Found by Sequence Length')

for ax, df, color, title in [
    (axes[0], df_web,   BLUE,     'Dataset A: Web'),
    (axes[1], df_purch, '#f97316', 'Dataset B: Purchases'),
]:
    len_counts = df.groupby('length').size()
    ax.bar(len_counts.index, len_counts.values, color=color, alpha=0.85, edgecolor='#18181b')
    ax.set_xlabel('Pattern length')
    ax.set_ylabel('Number of patterns')
    ax.set_title(title)
    ax.set_xticks(len_counts.index)
    ax.grid(axis='y')
    for x, y in zip(len_counts.index, len_counts.values):
        ax.text(x, y + 0.3, str(y), ha='center', fontsize=9, color=AMBER)

plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of the min_support threshold on the number of patterns
thresholds_web   = np.arange(0.10, 0.70, 0.05)
thresholds_purch = np.arange(0.10, 0.60, 0.05)

def count_patterns(db, thresholds, max_len=4):
    ps = PrefixSpan(db)
    counts = []
    for t in thresholds:
        mc = int(t * len(db))
        res = ps.frequent(mc, closed=False)
        counts.append(sum(1 for c, p in res if len(p) <= max_len))
    return counts

print('Computing sensitivity curves...')
counts_web   = count_patterns(web_db, thresholds_web)
counts_purch = count_patterns(purchase_db, thresholds_purch)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Sensitivity to the Support Threshold')

for ax, thrs, cnts, color, cur_t, title in [
    (axes[0], thresholds_web,   counts_web,   BLUE,     MIN_SUPPORT_WEB,   'Dataset A: Web'),
    (axes[1], thresholds_purch, counts_purch, '#f97316', MIN_SUPPORT_PURCH, 'Dataset B: Purchases'),
]:
    ax.plot(thrs, cnts, color=color, linewidth=2.5, marker='o', markersize=5)
    ax.fill_between(thrs, cnts, alpha=0.15, color=color)
    ax.axvline(cur_t, color=AMBER, linewidth=1.5, linestyle='--', label=f'current threshold = {cur_t}')
    ax.set_xlabel('min_support')
    ax.set_ylabel('Number of patterns (length <= 4)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

---
## 9. Business Analysis, Dataset A: Web Funnel

In web navigation, sequential patterns let us answer:
- What are the most frequent paths to `checkout`?
- Where do users get lost (drop-off)?
- Which intermediate pages increase the probability of conversion?

In [ ]:
# ── Conversion funnel analysis ───────────────────────────────────
FUNNEL = ['home', 'products', 'cart', 'checkout', 'payment', 'confirm']

print('CONVERSION FUNNEL: Retention rates per stage')
print('═' * 60)

funnel_data = []
for step in FUNNEL:
    count = sum(1 for s in web_db if step in s)
    sup   = count / len(web_db)
    funnel_data.append({'step': step, 'count': count, 'support': sup})
    print(f'  {step:12s}  {count:4d} sessions  ({sup:.0%})')

print()
print('Conversion rates between stages:')
for i in range(1, len(funnel_data)):
    prev  = funnel_data[i-1]
    curr  = funnel_data[i]
    rate  = curr['count'] / prev['count'] if prev['count'] > 0 else 0
    drop  = 1 - rate
    print(f'  {prev["step"]:12s} → {curr["step"]:12s}: '
          f'{rate:.0%} conversion  |  {drop:.0%} drop-off')

In [ ]:
# Visualize the funnel
df_funnel = pd.DataFrame(funnel_data)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_funnel['step'][::-1], df_funnel['support'][::-1],
               color=[WEB_COLORS.get(s, ZINC) for s in df_funnel['step'][::-1]],
               alpha=0.85, edgecolor='none', height=0.6)

for bar, row in zip(bars, df_funnel.iloc[::-1].itertuples()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{row.count} sessions ({row.support:.0%})',
            va='center', fontsize=9, color='#e4e4e7')

ax.set_xlabel('Fraction of total sessions')
ax.set_title('Web Conversion Funnel\n(% of sessions reaching each stage)')
ax.set_xlim(0, 1.3)
ax.grid(axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ── Paths to checkout ─────────────────────────────────────────────
checkout_paths = [
    (seq, ' → '.join(seq))
    for seq in web_db
    if 'checkout' in seq
]

# Group unique paths
from collections import Counter
path_counter = Counter(path_str for _, path_str in checkout_paths)

print(f'PATHS TO CHECKOUT ({len(checkout_paths)} sessions with checkout)')
print('═' * 65)
for path, count in path_counter.most_common(10):
    pct = count / len(web_db)
    bar = '█' * int(pct * 40)
    print(f'  {count:3d} ({pct:.0%}) {bar}')
    print(f'         {path}')
    print()

---
## 10. Business Analysis, Dataset B: Sequential Cross-sell

In purchases, sequential patterns let us:
- Design post-purchase **email sequences**
- Know when to recommend product B after the customer bought A
- Measure the effectiveness of **dynamic bundles** over time

In [ ]:
# ── Next-purchase recommendations ────────────────────────────────
def next_purchase_recommendations(db, seed_product, top_n=5):
    """Given that the customer has just bought 'seed_product',
       what should we recommend next?"""
    relevant = [seq for seq in db if seed_product in seq]
    next_counts = defaultdict(int)
    for seq in relevant:
        idx = seq.index(seed_product)
        if idx < len(seq) - 1:
            next_item = seq[idx + 1]
            if next_item != seed_product:
                next_counts[next_item] += 1
    total = len(relevant)
    recs = sorted(next_counts.items(), key=lambda x: -x[1])
    print(f'Seed: [{seed_product}]  ({len(relevant)} customers bought it)')
    print(f'Recommendations (immediate next purchase):')
    for item, cnt in recs[:top_n]:
        conf = cnt / total
        print(f'  -> {item:20s}  immediate confidence = {conf:.0%}  ({cnt} customers)')
    print()

for seed in ['laptop', 'phone', 'tablet']:
    next_purchase_recommendations(purchase_db, seed)
    print()

In [ ]:
# ── Full sequential confidence (not just the immediate next) ─────
print('SEQUENTIAL CONFIDENCE (gaps allowed)')
print('   conf([A] ⟹ [B]) = % of customers who bought A and later B (at any time)')
print('═' * 70)

seeds    = ['laptop', 'phone', 'tablet', 'charger']
targets  = ['mouse', 'keyboard', 'charger', 'case', 'earbuds', 'headphones']

conf_matrix = pd.DataFrame(index=seeds, columns=targets, dtype=float)
for s in seeds:
    for t in targets:
        conf_matrix.loc[s, t] = seq_confidence([s], t, purchase_db)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(conf_matrix.astype(float), ax=ax, cmap='YlOrRd', vmin=0, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 10},
            linewidths=0.3, linecolor='#18181b')
ax.set_title('Sequential Confidence: [seed product] ⟹ [eventual next purchase]')
ax.set_xlabel('Next purchase (eventual)')
ax.set_ylabel('Seed purchase')
plt.tight_layout()
plt.show()

print()
print('Reading: if the customer bought "laptop" there is a', end=' ')
v = conf_matrix.loc['laptop', 'mouse']
print(f'{v:.0%} probability that they buy "mouse" in some future purchase.')

In [ ]:
# ── Executive summary ─────────────────────────────────────────────
print('EXECUTIVE SUMMARY: BUSINESS RECOMMENDATIONS')
print('═' * 65)
print()
print('=== DATASET A: Web Navigation ===')
print()

# Largest drop-off
worst_drop = max(range(1, len(funnel_data)), key=lambda i:
    1 - (funnel_data[i]['count'] / funnel_data[i-1]['count'] if funnel_data[i-1]['count'] > 0 else 0))
prev = funnel_data[worst_drop-1]
curr = funnel_data[worst_drop]
drop = 1 - curr['count'] / prev['count']
print(f'  Largest drop in the funnel: {prev["step"]} -> {curr["step"]} ({drop:.0%} drop-off)')
print(f'     Recommendation: review the UX of the "{curr["step"]}" page')
print()

top_path = path_counter.most_common(1)[0]
print(f'  Most common conversion path:')
print(f'     {top_path[0]} ({top_path[1]} sessions)')
print()

print('=== DATASET B: Purchase History ===')
print()

# Top length-2 sequences by confidence in purchases
best_cross_sell = []
for _, row in df_purch[df_purch['length'] == 2].iterrows():
    a, b = row['pattern']
    conf = seq_confidence([a], b, purchase_db)
    best_cross_sell.append({'pattern': row['pattern_str'], 'conf': conf, 'sup': row['support']})

best_cross_sell = sorted(best_cross_sell, key=lambda x: -x['conf'])[:5]
print('  Top 5 sequential cross-sell opportunities:')
for i, item in enumerate(best_cross_sell, 1):
    print(f'     {i}. {item["pattern"]:35s} conf={item["conf"]:.0%}  sup={item["sup"]:.2f}')

---
## 11. Exercises

### Exercise 1: Order matters
Compare the support of `[home → products → cart]` with `[cart → products → home]`. Are they equal? Why would they get the same support in AR but not in SM?

### Exercise 2: Support threshold
For Dataset A, lower `MIN_SUPPORT_WEB` from 0.30 to 0.15. How many new patterns appear? Do you find patterns of 4+ events that are actionable from a business standpoint?

### Exercise 3: Funnel analysis
Which pages have the largest drop-off in Dataset A? If you were a UX analyst, what A/B experiment would you design to reduce that drop?

### Exercise 4: Cross-sell timing
Using Dataset B, compute the difference between **immediate confidence** (the very next purchase) and **sequential confidence** (any future purchase) for `laptop → keyboard`. Which one is more useful for an email campaign at D+7 after the purchase?

### Exercise 5: Your own dataset
Modify `NAVIGATION_PATTERNS` to simulate the **flow of a mobile fitness app** (onboarding → workout → stats → share → premium) and analyze the paths toward conversion to premium.